# AIoT Project - Time-Series Modeling

This notebook implements the raw/windowed time-series approach: load Protocol wrist IMU data from MongoDB, create fixed windows, filter the signal, flatten each window, apply scaling/PCA, and train/evaluate SVC and Random Forest classifiers.


In [ ]:
import os
import sys
from pathlib import Path

# Set random seeds for reproducibility
import random
import numpy as np
np.random.seed(42)
random.seed(42)

# basic data engineering
import pandas as pd
pd.options.mode.copy_on_write = True
import scipy

# plotting
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['figure.dpi'] = 100

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# db
import pymongo

# configs & other
import yaml
from tqdm.notebook import tqdm_notebook
from datetime import datetime
from time import time
import logging

from psynlig import pca_explained_variance_bar

# utils processing
from utils import sliding_window_pd
from utils import apply_filter
from utils import filter_instances
from utils import flatten_instances_df
from utils import df_rebase
from utils import rename_df_column_values

# utils visualization
from utils_visual import plot_instance_time_domain
from utils_visual import plot_instance_3d
from utils_visual import plot_np_instance
from utils_visual import plot_heatmap
from utils_visual import plot_scatter_pca

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

%load_ext autoreload
%autoreload 2

Start time of execution

In [ ]:
time_start = time()

## Load configuration

In [ ]:
config_path = os.path.join(os.getcwd(), "config.yml")

if not os.path.exists(config_path):
    logger.error(f"Config file not found at {config_path}")
    logger.info("Copy config.yml.template to config.yml and fill in your settings")
    raise FileNotFoundError(config_path)

try:
    with open(config_path) as file:
        config = yaml.load(file, Loader=yaml.FullLoader)
    logger.info(f"Configuration loaded successfully from {config_path}")
except yaml.YAMLError as e:
    logger.error(f"Failed to parse config.yml: {e}")
    raise

# Validate required config keys
required_keys = ["client", "db", "col", "sliding_window", "filter", "classifier", "fine_tune"]
missing_keys = [key for key in required_keys if key not in config]
if missing_keys:
    logger.error(f"Missing required config keys: {missing_keys}")
    raise KeyError(f"Config missing keys: {missing_keys}")

logger.info("Configuration validation passed")

In [ ]:
try:
    client = pymongo.MongoClient(config["client"], serverSelectionTimeoutMS=5000)
    # Verify connection
    client.admin.command("ping")
    logger.info(f"✓ Connected to MongoDB at {config['client']}")
except pymongo.errors.ConnectionFailure as e:
    logger.error(f"✗ Failed to connect to MongoDB: {e}")
    logger.info("Ensure MongoDB is running: mongod or mongodump service")
    raise
except Exception as e:
    logger.error(f"✗ Unexpected error connecting to MongoDB: {e}")
    raise

In [ ]:
db = client[config["db"]]
coll = db[config["col"]]

## Load data

Fetch the documents that correspond to the **sensor configuration you are evaluating**. Start with the **Protocol** split and the hand/wrist IMU in the AccGyr configuration.

```python
query = {"split": "Protocol", "imu_location": "hand", "sensor": "AccGyr"}
documents = list(coll.find(query))
```

The Optional split is useful for extra EDA, but several optional activities appear only for a subset of subjects. For the baseline model, Protocol is cleaner because every Protocol class has subject-disjoint train/test coverage.

Carry the `subject` field through every transformation (windowing, filtering, feature extraction) because the train/test split must be subject-disjoint.

In [ ]:
found_labels = coll.distinct("activity_label", {"split": "Protocol"})
print("Protocol activities in DB:", sorted(found_labels))

In [ ]:
query = {"split": "Protocol", "imu_location": "hand", "sensor": "AccGyr"}
documents = list(coll.find(query))

In [ ]:
print(f"Loaded documents: {len(documents)}")

if len(documents) > 0:
    example = documents[0]
    print("Example document keys:", sorted(example.keys()))
    print("Example sensor/imu_location/subject:", example.get("sensor"), example.get("imu_location"), example.get("subject"))

segments_df = pd.DataFrame([
    {
        "activity_label": doc["activity_label"],
        "activity_id": doc["activity_id"],
        "subject": doc["subject"],
        "imu_location": doc["imu_location"],
        "sensor": doc["sensor"],
        "segment_length": len(doc["data"]["acc_x"]),
        "sr": doc.get("sr", 100),
    }
    for doc in documents
])
segments_df["duration_sec"] = segments_df["segment_length"] / segments_df["sr"]
segments_df.head()

## Explore the nature of the data

Suggested exploratory plots for the PAMAP2 instances you loaded:

* Total recording time per activity (sum of segment lengths in seconds, grouped by `activity_label`).
* A time-domain plot of one segment per activity, so you can see the signal shape of each class.
* The distribution of segment counts per `(subject, activity_label)` pair — this exposes the class imbalance you will need to address.

In [ ]:
# Total recording time per activity
activity_duration = segments_df.groupby("activity_label")["duration_sec"].sum().sort_values(ascending=False)
plt.figure(figsize=(12, 5))
sns.barplot(x=activity_duration.index, y=activity_duration.values, hue=activity_duration.index, palette="viridis", legend=False)
plt.xticks(rotation=45, ha="right")
plt.title("Total recording time per activity - Protocol hand AccGyr")
plt.ylabel("Duration (s)")
plt.xlabel("Activity")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "activity_duration_protocol_hand_accgyr.png", bbox_inches="tight")
plt.show()

In [ ]:
# Time-domain plot of one segment per activity (first samples only for readability)
activity_examples = {}
for doc in documents:
    label = doc["activity_label"]
    if label not in activity_examples:
        activity_examples[label] = doc
    if len(activity_examples) == segments_df["activity_label"].nunique():
        break

plot_samples = 2000
for activity_label, doc in activity_examples.items():
    plt.figure(figsize=(16, 6))
    plot_instance_time_domain(pd.DataFrame(doc["data"]).head(plot_samples))
    plt.title(f"Time-domain segment for activity: {activity_label} (subject {doc['subject']}, first {plot_samples} samples)")
    plt.tight_layout()
    safe_label = activity_label.lower().replace(" ", "_").replace("/", "_")
    plt.savefig(RESULTS_DIR / f"time_domain_{safe_label}_protocol_hand_accgyr.png", bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
# Segment counts per activity and subject
plt.figure(figsize=(14, 6))
sns.countplot(data=segments_df, x="activity_label", hue="subject", palette="tab10")
plt.xticks(rotation=45, ha="right")
plt.title("Segment counts per activity and subject - Protocol hand AccGyr")
plt.ylabel("Segment count")
plt.xlabel("Activity")
plt.legend(title="Subject", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "segment_counts_protocol_hand_accgyr.png", bbox_inches="tight")
plt.show()

## Data Processing

* Apply the sliding window algorithm to each segment (use `sliding_window_pd` from `utils.py` with the parameters defined in `config.yml`).
* Apply a low-pass Butterworth filter to each window (use `apply_filter` / `filter_instances` from `utils.py`).
* Detect outliers and confirm that no `NaN` values survive from the ingestion step.

Keep `(window, activity_label, subject)` triplets together throughout this section — you need the subject ID for the train/test split.

In [ ]:
# Create sliding windows for each contiguous segment.
window_size = config["sliding_window"]["ws"]
overlap_ratio = config["sliding_window"]["overlap"]
step = int(window_size * (1 - overlap_ratio))
if step <= 0:
    step = window_size
print(f"Window size: {window_size}; step size: {step}; overlap ratio: {overlap_ratio}")

windowed_instances = []
for doc in tqdm_notebook(documents, desc="Creating sliding windows"):
    df_segment = pd.DataFrame(doc["data"])
    windows = sliding_window_pd(
        df_segment,
        ws=window_size,
        overlap=step,
        w_type=config["sliding_window"]["w_type"],
        w_center=config["sliding_window"]["w_center"],
        print_stats=False,
    )
    for window in windows:
        windowed_instances.append({
            "window": window,
            "activity_label": doc["activity_label"],
            "activity_id": doc["activity_id"],
            "subject": doc["subject"],
        })

print(f"Created {len(windowed_instances)} windows from {len(documents)} segments.")

In [ ]:
# Apply filtering to each window and store the filtered windows in a new list
filtered_windows = filter_instances(
    [item["window"] for item in windowed_instances],
    order=config["filter"]["order"],
    wn=config["filter"]["wn"],
    filter_type=config["filter"]["type"],
)

# Combine filtered windows with their corresponding labels and subjects into a new list of instances
filtered_instances = [
    {
        "window": filtered_windows[i],
        "activity_label": windowed_instances[i]["activity_label"],
        "activity_id": windowed_instances[i]["activity_id"],
        "subject": windowed_instances[i]["subject"],
    }
    for i in range(len(filtered_windows))
]
print(f"Filtered {len(filtered_instances)} windows.")

In [ ]:
# Check for NaN values in the filtered windows
nan_after_filter = [inst["window"].isna().sum().sum() for inst in filtered_instances]
print("NaN values per filtered window (sample):", nan_after_filter[:10])
assert not any(nan_after_filter), "There are NaN values after filtering!"

processed_instances = pd.DataFrame([
    {
        "activity_label": inst["activity_label"],
        "activity_id": inst["activity_id"],
        "subject": inst["subject"],
        "window": inst["window"],
    }
    for inst in filtered_instances
])
print(processed_instances.shape)

plt.figure(figsize=(12, 5))
sns.countplot(data=processed_instances, x="activity_label", hue="activity_label", palette="viridis", legend=False)
plt.xticks(rotation=45, ha="right")
plt.title("Window count per activity after segmentation - Protocol hand AccGyr")
plt.ylabel("Window count")
plt.xlabel("Activity")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "window_counts_protocol_hand_accgyr.png", bbox_inches="tight")
plt.show()

## Train/Test split

The split must be **by subject**: 6–7 subjects in train, 2–3 in test, with **no overlap**. This forces the model to generalize to users it has never seen, which is the realistic deployment scenario.

Do **not** use `train_test_split` with random shuffling — that would leak windows from the same subject into both sets and inflate the reported metrics.

In [ ]:
# Train-test split by subject 

X_train, y_train = [], []
X_test, y_test = [], []


In [ ]:
TRAIN_SUBJECTS = ["101", "102", "103", "104", "105", "107"]
TEST_SUBJECTS  = ["106", "108", "109"]

assert set(TRAIN_SUBJECTS).isdisjoint(TEST_SUBJECTS)

train_labels = set(processed_instances.loc[processed_instances["subject"].isin(TRAIN_SUBJECTS), "activity_id"])
test_labels = set(processed_instances.loc[processed_instances["subject"].isin(TEST_SUBJECTS), "activity_id"])
print("Train labels:", sorted(train_labels))
print("Test labels:", sorted(test_labels))
print("Labels missing from train:", sorted(test_labels - train_labels))
print("Labels missing from test:", sorted(train_labels - test_labels))

In [ ]:
# Raw/windowed time-series representation: flatten each filtered window.
X_train_raw, y_train_raw = [], []
X_test_raw, y_test_raw = [], []

for _, inst in processed_instances.iterrows():
    subject = inst["subject"]
    features = inst["window"].to_numpy().flatten()
    label = inst["activity_id"]

    if subject in TRAIN_SUBJECTS:
        X_train_raw.append(features)
        y_train_raw.append(label)
    elif subject in TEST_SUBJECTS:
        X_test_raw.append(features)
        y_test_raw.append(label)

X_train_raw = np.vstack(X_train_raw)
X_test_raw = np.vstack(X_test_raw)
y_train_raw = np.array(y_train_raw)
y_test_raw = np.array(y_test_raw)

# Keep the old variable names for the raw/PCA baseline cells below.
X_train, X_test = X_train_raw, X_test_raw
y_train, y_test = y_train_raw, y_test_raw

print("Raw train shape:", X_train_raw.shape)
print("Raw test shape:", X_test_raw.shape)

## Scaling

Fit the scaler on the training subjects only, then apply it to the test subjects.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
# Scaling the features using StandardScaler (zero mean, unit variance)
scaler = StandardScaler()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# Fit the scaler on the training data and transform both training and test data
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Scaled feature shape: {X_train.shape}")

## Dimensionality Reduction (PCA)

Apply PCA to reduce the feature dimensionality while retaining most of the variance. This is especially useful for high-dimensional windowed sensor data.



In [ ]:
from sklearn.decomposition import PCA

# Apply PCA with 99% variance retention (or adjust n_components based on your analysis)
n_components = config["PCA"]["n_comp"]  

print(f"\nApplying PCA with {n_components} components (99% variance retention)...")

pca = PCA(n_components=n_components, random_state=42)

# Fit PCA on training data only (avoid data leakage)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"PCA-reduced feature shape: {X_train_pca.shape}")
print(f"Dimensionality reduction: {X_train.shape[1]} → {X_train_pca.shape[1]} features ({100*X_train_pca.shape[1]/X_train.shape[1]:.1f}%)")
print(f"Variance retained: {pca.explained_variance_ratio_.sum():.4f}")


In [ ]:
from sklearn.decomposition import PCA

# Analyze explained variance with all components
pca_full = PCA()
pca_full.fit(X_train)

# Cumulative explained variance
cumsum_var = np.cumsum(pca_full.explained_variance_ratio_)

# Plot explained variance
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(pca_full.explained_variance_ratio_) + 1), 
         pca_full.explained_variance_ratio_, 'bo-', linewidth=2)
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.title("Scree Plot: Explained Variance per Component")
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumsum_var) + 1), cumsum_var, 'ro-', linewidth=2)
plt.axhline(y=0.85, color='g', linestyle='--', label='85% variance')
plt.axhline(y=0.95, color='b', linestyle='--', label='95% variance')
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Determine optimal number of components to retain 95% variance
n_components_95 = np.argmax(cumsum_var >= 0.95) + 1
n_components_85 = np.argmax(cumsum_var >= 0.85) + 1

print(f"Original feature dimension: {X_train.shape[1]}")
print(f"Components to retain 85% variance: {n_components_85}")
print(f"Components to retain 95% variance: {n_components_95}")
print(f"Variance retained with {n_components_95} components: {cumsum_var[n_components_95-1]:.4f}")


### Apply simple classifier

In [ ]:
# Use PCA-reduced features
print(f"Using PCA-Reduced Features: {X_train_pca.shape[1]} features")

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Train classifiers on PCA-reduced features

# Hyperparameters from config for SVC
kernel = config["classifier"]["SVC"]["kernel"]
C = config["classifier"]["SVC"]["C"]
gamma = config["classifier"]["SVC"]["gamma"]
class_weight = config["classifier"]["SVC"].get("class_weight", None)

# Hyperparameters from config for Random Forest
n_estimators = config["classifier"]["RandomForest"]["n_estimators"]
max_depth = config["classifier"]["RandomForest"]["max_depth"]

print("Training on PCA-Reduced Features...")

# SVM classifier
print("Training SVC:")
svm = SVC(kernel=kernel, C=C, gamma=gamma, class_weight=class_weight, random_state=42)
svm.fit(X_train_pca, y_train)

# Random Forest classifier
print("Training Random Forest:")
rf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
rf.fit(X_train_pca, y_train)


### Evaluate simple classifier

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

summary_results = []

def evaluate_model(name, model, X_test_eval, y_test_eval, result_rows, labels=None, save_cm=None):
    y_pred = model.predict(X_test_eval)
    acc = accuracy_score(y_test_eval, y_pred)
    precision = precision_score(y_test_eval, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_test_eval, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test_eval, y_pred, average="weighted", zero_division=0)

    print(f"\n{name}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(classification_report(y_test_eval, y_pred, zero_division=0))

    result_rows.append({
        "Approach": name,
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
    })

    if save_cm is not None:
        disp = ConfusionMatrixDisplay.from_predictions(y_test_eval, y_pred, display_labels=labels, xticks_rotation=45, cmap="Blues")
        disp.ax_.set_title(name)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / save_cm, bbox_inches="tight")
        plt.show()

    return y_pred

print("\nEVALUATION: Raw Windowed PCA Models")
print("=" * 60)
raw_label_order = sorted(np.unique(np.concatenate([y_train, y_test])))

evaluate_model("Raw PCA SVC", svm, X_test_pca, y_test, summary_results, labels=raw_label_order, save_cm="cm_raw_pca_svc.png")
evaluate_model("Raw PCA Random Forest", rf, X_test_pca, y_test, summary_results, labels=raw_label_order, save_cm="cm_raw_pca_random_forest.png")

results_df = pd.DataFrame(summary_results)
print("\nSUMMARY TABLE")
print(results_df.to_string(index=False))

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
from sklearn.metrics import classification_report

### Apply optimization with Grid Search and/or Cross-validation

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

In [ ]:
def run_grid_search(name, model, param_grid, X_train, y_train, cv, verbose):
    print(f"\n GRID SEARCH: {name}\n")

    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=config["fine_tune"]["grid_search"]["scoring"],
        refit=True,
        cv=cv,
        verbose=verbose,
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    print("\nBEST RESULT")
    print("Params:", grid.best_params_)
    print("F1:", round(grid.best_score_, 4))

    return grid


### Evaluate optimized classifier

Once you are satisfied with the wrist-only configuration, repeat the **load → process → train → evaluate** flow for the chest and ankle IMUs (and combinations thereof) and compare the metrics in your report.

In [ ]:
RUN_GRID_SEARCH = False

if RUN_GRID_SEARCH:
    print("Evaluation of Grid Search results on test set:")

    grid = run_grid_search(
        name="SVC",
        model=SVC(random_state=42),
        param_grid=config["fine_tune"]["SVC"]["param_grid"],
        X_train=X_train_pca,
        y_train=y_train,
        cv=config["fine_tune"]["SVC"]["cv"],
        verbose=config["fine_tune"]["SVC"]["verbose"],
    )

    best_svc = SVC(**grid.best_params_, random_state=42)
    best_svc.fit(X_train_pca, y_train)
    y_pred_svc = best_svc.predict(X_test_pca)

    print("\nClassification Report for Best SVC Model:")
    print(classification_report(y_test, y_pred_svc, zero_division=0))

    cm_svc = confusion_matrix(y_test, y_pred_svc)
    print("Confusion Matrix for Best SVC Model:")
    print(cm_svc)

    disp_svc = ConfusionMatrixDisplay(confusion_matrix=cm_svc)
    disp_svc.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix - Best SVC Model")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "cm_grid_best_svc.png", bbox_inches="tight")
    plt.show()

    grid = run_grid_search(
        name="RandomForest",
        model=RandomForestClassifier(random_state=42),
        param_grid=config["fine_tune"]["RandomForest"]["param_grid"],
        X_train=X_train_pca,
        y_train=y_train,
        cv=config["fine_tune"]["RandomForest"]["cv"],
        verbose=config["fine_tune"]["RandomForest"]["verbose"],
    )

    best_rf = RandomForestClassifier(**grid.best_params_, random_state=42)
    best_rf.fit(X_train_pca, y_train)
    y_pred_rf = best_rf.predict(X_test_pca)

    print("\nClassification Report for Best Random Forest Model:")
    print(classification_report(y_test, y_pred_rf, zero_division=0))

    cm_rf = confusion_matrix(y_test, y_pred_rf)
    print("Confusion Matrix for Best Random Forest Model:")
    print(cm_rf)

    disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf)
    disp_rf.plot(cmap=plt.cm.Oranges)
    plt.title("Confusion Matrix - Best Random Forest Model")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "cm_grid_best_random_forest.png", bbox_inches="tight")
    plt.show()
else:
    print("Grid Search skipped. Set RUN_GRID_SEARCH = True when the baseline and feature pipelines are stable.")